# General Data Analysis

This notebook scans the contents of `data/`, loads each table, and runs a reusable exploratory analysis workflow.

It includes:
- dataset inventory and quick previews
- missingness, numeric, and categorical summaries for every table
- lightweight plotting helpers
- a focused section for `players_quarters_final.csv`, which is the main contest dataset


In [ ]:
from pathlib import Path
from typing import Dict

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
plt.style.use("ggplot")

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
assert DATA_DIR.exists(), f"Missing data directory: {DATA_DIR}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")


In [ ]:
def load_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix == ".parquet":
        df = pd.read_parquet(path)
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    for col in df.select_dtypes(include="object").columns:
        non_null = df[col].dropna()
        if non_null.empty:
            continue
        values = set(non_null.astype(str).str.strip().str.lower().unique())
        if values <= {"true", "false"}:
            df[col] = df[col].astype(str).str.strip().str.lower().map({"true": True, "false": False})
    return df


def build_inventory(data_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(data_dir.iterdir()):
        if not path.is_file():
            continue
        if path.suffix.lower() not in {".csv", ".parquet", ".xlsx", ".xls"}:
            continue
        df = load_table(path)
        rows.append(
            {
                "file_name": path.name,
                "format": path.suffix.lower().lstrip("."),
                "rows": len(df),
                "columns": df.shape[1],
                "size_mb": round(path.stat().st_size / (1024 ** 2), 3),
            }
        )
    return pd.DataFrame(rows).sort_values(["rows", "file_name"], ascending=[False, True]).reset_index(drop=True)


def dataset_overview(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=True),
    })
    return out.sort_values(["missing_pct", "unique"], ascending=[False, False])


def numeric_summary(df: pd.DataFrame) -> pd.DataFrame:
    num = df.select_dtypes(include=np.number)
    if num.empty:
        return pd.DataFrame()
    summary = num.describe().T
    summary["missing"] = df[num.columns].isna().sum()
    summary["missing_pct"] = (df[num.columns].isna().mean() * 100).round(2)
    return summary.sort_values("std", ascending=False)


def categorical_summary(df: pd.DataFrame, top_n: int = 8) -> Dict[str, pd.DataFrame]:
    result = {}
    cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()
    for col in cat_cols:
        counts = df[col].fillna("<NA>").value_counts(dropna=False).head(top_n)
        result[col] = pd.DataFrame({
            "count": counts,
            "pct": (counts / len(df) * 100).round(2),
        })
    return result


def plot_top_missing(df: pd.DataFrame, title: str, top_n: int = 15) -> None:
    missing = (df.isna().mean() * 100).sort_values(ascending=False).head(top_n)
    if missing.max() == 0:
        print("No missing values in this table.")
        return
    ax = missing.sort_values().plot(kind="barh", figsize=(8, max(4, top_n * 0.35)), color="#C44E52")
    ax.set_title(f"Top Missingness: {title}")
    ax.set_xlabel("Missing values (%)")
    ax.set_ylabel("")
    plt.show()


def plot_numeric_distributions(df: pd.DataFrame, title: str, max_cols: int = 6) -> None:
    num_cols = df.select_dtypes(include=np.number).columns.tolist()[:max_cols]
    if not num_cols:
        print("No numeric columns to plot.")
        return
    ncols = 2
    nrows = math.ceil(len(num_cols) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, num_cols):
        df[col].dropna().plot(kind="hist", bins=30, ax=ax, color="#4C72B0", alpha=0.8)
        ax.set_title(col)
    for ax in axes[len(num_cols):]:
        ax.axis("off")
    fig.suptitle(f"Numeric Distributions: {title}", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
inventory = build_inventory(DATA_DIR)
tables = {
    path.name: load_table(path)
    for path in sorted(DATA_DIR.iterdir())
    if path.is_file() and path.suffix.lower() in {".csv", ".parquet", ".xlsx", ".xls"}
}

display(Markdown("## Data Inventory"))
display(inventory)

display(Markdown("## Quick Preview"))
for name, df in tables.items():
    display(Markdown(f"### `{name}`"))
    print(f"shape = {df.shape}")
    display(df.head())


In [ ]:
for name, df in tables.items():
    display(Markdown(f"## Overview: `{name}`"))
    display(dataset_overview(df))

    num_summary = numeric_summary(df)
    if not num_summary.empty:
        display(Markdown("### Numeric Summary"))
        display(num_summary)

    cat_summary = categorical_summary(df)
    if cat_summary:
        display(Markdown("### Top Categories"))
        for col, summary in cat_summary.items():
            display(Markdown(f"**{col}**"))
            display(summary)

    plot_top_missing(df, name)
    plot_numeric_distributions(df, name)


## Focused Analysis: `players_quarters_final.csv`

The contest documentation describes this as the main analytical dataset. The cells below examine class balance, time-checkpoint structure, positional differences, and simple feature relationships with `scored_after`.


In [ ]:
main_df = tables["players_quarters_final.csv"].copy()
main_df["date"] = pd.to_datetime(main_df["date"])

display(Markdown("### Main Dataset Shape"))
print(main_df.shape)

display(Markdown("### Target Balance"))
target_summary = main_df["scored_after"].value_counts().rename_axis("scored_after").to_frame("count")
target_summary["pct"] = (target_summary["count"] / len(main_df) * 100).round(2)
display(target_summary)

display(Markdown("### Checkpoint Coverage"))
display(main_df["checkpoint"].value_counts().to_frame("count"))

display(Markdown("### Position x Target Rate"))
position_target = main_df.groupby("position")["scored_after"].agg(["count", "mean"])
position_target["mean"] = (position_target["mean"] * 100).round(2)
position_target = position_target.rename(columns={"mean": "scored_after_rate_pct"})
display(position_target.sort_values("scored_after_rate_pct", ascending=False))

display(Markdown("### Checkpoint x Target Rate"))
checkpoint_target = main_df.groupby("checkpoint")["scored_after"].agg(["count", "mean"])
checkpoint_target["mean"] = (checkpoint_target["mean"] * 100).round(2)
checkpoint_target = checkpoint_target.rename(columns={"mean": "scored_after_rate_pct"})
display(checkpoint_target.sort_index())


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

main_df["checkpoint"].value_counts().sort_index().plot(kind="bar", ax=axes[0, 0], color="#4C72B0")
axes[0, 0].set_title("Rows per Checkpoint")
axes[0, 0].set_xlabel("")

main_df.groupby("checkpoint")["scored_after"].mean().sort_index().mul(100).plot(kind="bar", ax=axes[0, 1], color="#55A868")
axes[0, 1].set_title("Scored After Rate by Checkpoint (%)")
axes[0, 1].set_xlabel("")

main_df.groupby("position")["scored_after"].mean().mul(100).sort_values(ascending=False).plot(kind="bar", ax=axes[1, 0], color="#C44E52")
axes[1, 0].set_title("Scored After Rate by Position (%)")
axes[1, 0].set_xlabel("")

main_df.groupby("is_home")["scored_after"].mean().mul(100).plot(kind="bar", ax=axes[1, 1], color="#8172B2")
axes[1, 1].set_title("Scored After Rate by Home/Away (%)")
axes[1, 1].set_xlabel("is_home")

plt.tight_layout()
plt.show()


In [ ]:
feature_cols = [
    col for col in main_df.columns
    if col.startswith("last15_") or col.startswith("cumul_")
]

corr = main_df[feature_cols + ["scored_after"]].corr(numeric_only=True)["scored_after"]
corr = corr.drop("scored_after").sort_values(key=lambda s: s.abs(), ascending=False)

display(Markdown("### Strongest Linear Relationships with `scored_after`"))
display(corr.to_frame("correlation_with_scored_after").head(15))

top_features = corr.head(6).index.tolist()
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.ravel()
for ax, col in zip(axes, top_features):
    main_df.boxplot(column=col, by="scored_after", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("scored_after")
for ax in axes[len(top_features):]:
    ax.axis("off")
plt.suptitle("Top Feature Distributions by Target", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
event_tables = [
    "player_appearance_pass.csv",
    "player_appearance_behaviour_under_pressure.csv",
    "player_appearance_run.csv",
    "player_appearance_shot_limited.csv",
]

event_counts = []
for name in event_tables:
    event_df = tables[name]
    per_player = event_df.groupby("player_appearance_id").size()
    event_counts.append(
        {
            "table": name,
            "rows": len(event_df),
            "unique_player_appearances": event_df["player_appearance_id"].nunique(),
            "avg_events_per_player_appearance": round(per_player.mean(), 2),
            "median_events_per_player_appearance": round(per_player.median(), 2),
        }
    )

display(Markdown("### Event Table Coverage"))
display(pd.DataFrame(event_counts).sort_values("rows", ascending=False))

run_events = tables["player_appearance_run.csv"]
run_summary = (
    run_events.groupby("run_type")[["distance", "max_speed"]]
    .agg(["count", "mean", "median", "max"])
    .round(2)
)
display(Markdown("### Run Event Summary by Type"))
display(run_summary)

shots = tables["player_appearance_shot_limited.csv"]
display(Markdown("### Shot Context"))
display(pd.crosstab(shots["stage"], shots["under_pressure"], margins=True))

passes = tables["player_appearance_pass.csv"]
pass_accuracy = (
    passes.groupby("stage")["accurate"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "accuracy_rate"})
)
pass_accuracy["accuracy_rate"] = (pass_accuracy["accuracy_rate"] * 100).round(2)
display(Markdown("### Pass Accuracy by Stage"))
display(pass_accuracy.sort_values("accuracy_rate", ascending=False))


## Suggested Next Steps

- Add domain-specific features from pass and pressure events back into the main checkpoint panel.
- Check whether class imbalance handling is needed before modeling `scored_after`.
- Compare behavior across positions, formations, and match periods.
- Extend the notebook with feature engineering and baseline models once the EDA looks sound.
